In [ ]:
# Package path adaptation only: keep all legacy scientific calculations unchanged.
from pathlib import Path as _PackagePath
import os as _package_os
_package_candidates = (_PackagePath.cwd(), _PackagePath.cwd().parent, _PackagePath.cwd() / "single_period")
_package_root = next((p for p in _package_candidates if (p / "notebooks").is_dir() and (p / "pinn_results").is_dir()), None)
if _package_root is None:
    raise RuntimeError("Start this notebook from the single_period package or one of its immediate subdirectories.")
_package_os.chdir(_package_root)
(_package_root / "generated" / "one-period-result-fig").mkdir(parents=True, exist_ok=True)
(_package_root / "generated" / "comparison_results").mkdir(parents=True, exist_ok=True)
import torch
import torch.nn as nn
import numpy as np
import os
import time
from torch.distributions.normal import Normal
import matplotlib.pyplot as plt
from tqdm import tqdm  # 导入进度条库
%matplotlib inline

# ================= 路径配置 =================
RESULTS_ROOT = "./generated/pinn_results_alphabeta1"  

# ================= 参数扫描配置 =================
# 模式1：固定tau，扫描alpha和beta
SCAN_ALPHA_BETA = True
tau_fixed = 3.0  # 固定tau值
# beta_list = np.arange(0.2, 1.3, 0.1).tolist()
# alpha_list = np.arange(0.2, 1.3, 0.1).tolist()
# alpha_list = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
# beta_list = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2]
alpha_list = np.linspace(0.2, 1.2, 51)
beta_list = np.linspace(0.2, 1.2, 51)
# beta_list = [0.5, 0.8, 1.2]
# alpha_list = [0.8, 1.0, 1.2]

# 模式2：固定alpha和beta，扫描tau
SCAN_TAU = False
alpha_fixed = 1.2
beta_fixed = 0.5
# tau_list = [0.5,1.0,1.5,2.0,2.5,3.0, 3.5, 4.0]
# tau_list = [2.0,2.5, 3.5, 4.0]
tau_list=[3.6,3.7,3.8,3.85,3.9]


# 生成参数组合
param_combinations = []

if SCAN_ALPHA_BETA:
    for beta_val in beta_list:
        for alpha_val in alpha_list:
            param_combinations.append({
                'beta': beta_val,
                'alpha': alpha_val,
                'tau': tau_fixed,
                'scan_type': 'alpha_beta'
            })

if SCAN_TAU:
    for tau_val in tau_list:
        param_combinations.append({
            'beta': beta_fixed,
            'alpha': alpha_fixed,
            'tau': tau_val,
            'scan_type': 'tau'
        })


# 固定参数
pi = 100.0   # 参数 pi
r = 0.05     # 无风险利率
T = 4.0      # 资产价格相关参数
K = 50.0     # 行权价
SEED = 100

print(f"Total parameter combinations to run: {len(param_combinations)}")

# ================= 主循环 =================
for i, params in enumerate(tqdm(param_combinations, desc="Overall Progress")):
    beta = params['beta']
    alpha = params['alpha']
    tau = params['tau']
    
    print(f"\n{'='*60}")
    print(f"Starting simulation {i+1}/{len(param_combinations)}: beta={beta}, alpha={alpha}, tau={tau}")
    print(f"{'='*60}\n")
    
    # 创建带参数的子文件夹
    param_str = f"beta_{beta}_alpha_{alpha}_tau_{tau}"
    timestamp = time.strftime("%Y%m%d-%H%M%S")
    RUN_DIR = f"run_{param_str}_{timestamp}"
    SAVE_DIR = os.path.join(RESULTS_ROOT, RUN_DIR)
    os.makedirs(SAVE_DIR, exist_ok=True)
    print(f"Results will be saved to: {SAVE_DIR}")

    # ================= 设置随机种子 =================
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    normal = Normal(0, 1)  # 标准正态分布

    # ================= 神经网络定义 =================
    class CarbonOptionNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(2, 16),
                nn.Tanh(),
                nn.Linear(16, 32),
                nn.Tanh(),
                nn.Linear(32, 16),
                nn.Tanh(),
                nn.Linear(16, 1),
            )

        def forward(self, A, t):
            if A.dim() == 1:
                A = A.view(-1, 1)
            if t.dim() == 1:
                t = t.view(-1, 1)
            x = torch.cat([A, t], dim=1)
            return self.net(x)

    # ================= 工具函数 =================
    def derivative(output, input_var, order=1):
        if output.dim() == 1:
            output = output.view(-1, 1)
        
        grad = output
        for _ in range(order):
            grad = torch.autograd.grad(
                outputs=grad,
                inputs=input_var,
                grad_outputs=torch.ones_like(grad),
                create_graph=True,
                retain_graph=True,
            )[0]
            
            if grad is None:
                grad = torch.zeros_like(input_var)
        
        return grad

    def dfdt(model, A, t, order=1):
        t_requires_grad = t.requires_grad
        if not t_requires_grad:
            t.requires_grad_(True)
        
        C = model(A, t)
        dC_dt = derivative(C, t, order)
        
        if not t_requires_grad:
            t.requires_grad_(False)
        
        return dC_dt

    def dfdx(model, A, t, order=1):
        A_requires_grad = A.requires_grad
        if not A_requires_grad:
            A.requires_grad_(True)
        
        C = model(A, t)
        dC_dA = derivative(C, A, order)
        
        if not A_requires_grad:
            A.requires_grad_(False)
        
        return dC_dA

    def final_condition(A, K):
        if A.dim() == 1:
            A = A.view(-1, 1)
        return torch.maximum(A - K, torch.tensor(0.0, device=A.device)).reshape(-1, 1)

    # ================= PDE损失函数 =================
    def pde_loss(model, A, t, pi, beta, alpha, r, T, tau):
        if A.dim() == 1:
            A = A.view(-1, 1).requires_grad_(True)
        if t.dim() == 1:
            t = t.view(-1, 1).requires_grad_(True)
        
        dC_dt = dfdt(model, A, t, order=1)
        d2C_dA2 = dfdx(model, A, t, order=2)
        
        z_t = beta * (T - t).pow(-alpha)
        
        A_pi_ratio = A / pi
        A_pi_ratio = A_pi_ratio.clamp(min=1e-6, max=1-1e-6)
        
        Phi_inv = normal.icdf(A_pi_ratio)
        Phi_prime = torch.exp(-0.5 * Phi_inv.pow(2)) / torch.sqrt(torch.tensor(2.0 * torch.pi))
        
        nonlinear_term = 0.5 * d2C_dA2 * (pi * Phi_prime * torch.sqrt(z_t)).pow(2)
        
        pde_residual = dC_dt + nonlinear_term - r * model(A, t)
        
        return pde_residual.pow(2).mean()

    # ================= 总损失函数 =================
    def compute_loss(model, A, t, A_domain, t_domain, pi, beta, alpha, r, T, tau, K):
        pde_loss_val = pde_loss(model, A, t, pi, beta, alpha, r, T, tau)
        
        if t_domain.dim() == 1:
            t_domain = t_domain.view(-1, 1)
        if A_domain.dim() == 1:
            A_domain = A_domain.view(-1, 1)
        
        # 边界A=0条件
        boundary_A0 = torch.ones_like(t_domain) * A_domain[0]
        boundary_loss_A0 = model(boundary_A0, t_domain)
        
        # 边界A=max条件
        boundary_A1 = torch.ones_like(t_domain) * A_domain[-1]
        discount = torch.exp(-r * (tau - t_domain))
        boundary_loss_A1 = model(boundary_A1, t_domain) - (A_domain[-1] - K * discount)
        
        # 终端条件 (t=tau)
        final_weight = 50.0
        f_final = final_condition(A_domain, K)
        t_final = torch.ones_like(A_domain) * tau
        
        if not A_domain.requires_grad:
            A_domain.requires_grad_(True)
        
        final_loss = model(A_domain, t_final) - f_final
        
        loss_val = (
            10 * pde_loss_val +
            boundary_loss_A0.pow(2).mean() +
            # boundary_loss_A1.pow(2).mean() +
            final_weight * final_loss.pow(2).mean()
        )
        
        return loss_val

    # ================= 数据准备 =================
    A_min, A_max = 0.0, 100.0
    t_min, t_max = 0.0, tau
    n_points = 150

    A_domain_tensor = torch.linspace(A_min, A_max, n_points, requires_grad=True, device=device)
    t_domain_tensor = torch.linspace(t_min, t_max, n_points, requires_grad=True, device=device)

    A_grid, t_grid = torch.meshgrid(A_domain_tensor, t_domain_tensor, indexing='ij')
    A_flat = A_grid.reshape(-1, 1)
    t_flat = t_grid.reshape(-1, 1)

    # ================= 模型训练 =================
    model = CarbonOptionNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    train_loss = []
    print("Starting training...")
    start_time = time.time()

    for epoch in range(15001):
        optimizer.zero_grad()
        
        loss = compute_loss(
            model, 
            A_flat.detach().requires_grad_(True), 
            t_flat.detach().requires_grad_(True),
            A_domain_tensor,
            t_domain_tensor,
            pi, beta, alpha, r, T, tau, K
        )
        
        loss.backward()
        optimizer.step()
        
        train_loss.append(loss.item())
        if epoch % 2000 == 0:
            print(f"Epoch {epoch:4d}: Loss = {loss.item():.8f}")

    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")

    # ================= 保存结果 =================
    loss_data = np.array(train_loss)
    loss_path = os.path.join(SAVE_DIR, "training_loss.npy")
    np.save(loss_path, loss_data)

    with torch.no_grad():
        predictions = model(A_flat, t_flat)
        predictions = predictions.cpu().numpy()

    A_vals = A_flat.cpu().detach().numpy()
    t_vals = t_flat.cpu().detach().numpy()

    results = np.hstack((A_vals, t_vals, predictions))
    results_path = os.path.join(SAVE_DIR, "pinn_results.npy")
    np.save(results_path, results)

    model_path = os.path.join(SAVE_DIR, "carbon_option_model.pth")
    torch.save(model.state_dict(), model_path)

    # 保存参数信息
    params_info = f"""
Model Parameters:
----------------
Pi = {pi}
Beta = {beta}
Alpha = {alpha}
Risk-free rate (r) = {r}
T = {T}
Tau = {tau}
Strike price (K) = {K}

Training Parameters:
-------------------
Points per dimension = {n_points}
Epochs = {15001}
Learning rate = {1e-3}
Training time = {training_time:.2f} seconds
"""

    params_path = os.path.join(SAVE_DIR, "simulation_parameters.txt")
    with open(params_path, "w") as f:
        f.write(params_info)

    print(f"Results saved to: {SAVE_DIR}")

print("\nAll simulations completed!")